### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="delays_zurich_transport",
    dataset_year="2017",
    domain_str="business & marketing",
    # Data Source
    dataset_source="OpenML",
    original_dataset_source_download_link="https://www.openml.org/search?type=data&sort=runs&status=active&id=40753",
    download_description="""
    In this notebook, run:

    import openml

    # Load the dataset object from OpenML
    dataset = openml.datasets.get_dataset(
        40753,
        download_data=True,
        download_qualities=False,
        download_features_meta_data=False,
        )
    dataset.get_data()[0].to_csv("../../../local-data-warehouse/delays_zurich_transport/delays_zurich_transport.csv", index=False)

""",
    # References
    academic_reference_bibtex=r"""@misc{seibold2017delayszurichtransport,
    author       = {Heidi Seibold},
    title        = {delays\_zurich\_transport},
    year         = {2017},
    month        = jun,
    howpublished = {OpenML dataset 40753},
    note         = {Uploaded 2017-06-01},
    url          = {https://www.openml.org/d/40753}
    }

""",
    academic_reference_bibtex_key="heidi2017delayszurichtransport",
    license="CCZero",
    data_tags=["Temporal"],
    curation_comments="""
    - We assign categorical data types to the columns "line_number" and "stop_id" since they represent discrete categories rather than continuous numerical values. 

    
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="delay",
    problem_type="regression",
    objective_metric_name="rmse",
    time_on="time",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

'''
Thoughts:
- Weather features leak, need to use lags
- 29 days are given, most with 130k - 213k rows. Could define three splits, one for each week.

'''

cat_features_non_object = ["line_number", "stop_id"]

df = pd.read_csv(dataset_mold.path / "delays_zurich_transport.csv", engine="pyarrow")

# df["time"] = pd.to_datetime(df["time"], errors="coerce")

print("Loaded data shape:", df.shape)

Loaded data shape: (5465575, 15)


In [ ]:
import pandas as pd

# transit_df: one row per transit event
# weather_df: one row per day with full-day aggregates

transit_df["date"] = pd.to_datetime(transit_df["date"]).dt.normalize()
weather_df["date"] = pd.to_datetime(weather_df["date"]).dt.normalize()

# shift weather forward so weather from d-1 is attached to transit day d
weather_lag1 = weather_df.copy()
weather_lag1["date"] = weather_lag1["date"] + pd.Timedelta(days=1)

weather_cols = [
    "temp", "windspeed_max", "windspeed_avg",
    "precipitation", "dew_point", "humidity"
]

rename_map = {c: f"{c}_lag1d" for c in weather_cols}
weather_lag1 = weather_lag1.rename(columns=rename_map)

df = transit_df.merge(
    weather_lag1[["date"] + list(rename_map.values())],
    on="date",
    how="left"
)

In [ ]:
unique_dates = df.time.dt.round("1D").unique()
unique_dates.day_of_week

array([6, 0, 1, 2, 3, 4, 5, 6, 0, 1, 2, 3, 4, 5, 6, 6, 0, 1, 2, 3, 4, 5,
       6, 0, 1, 2, 3, 4, 5], dtype=int32)

In [43]:
df.time.dt.day_of_year.value_counts()

time
327    213911
307    213510
308    213348
326    212994
306    212346
322    211917
305    211665
320    210643
321    210560
319    210524
316    210230
315    209899
309    209478
314    208544
323    207697
328    206898
330    205442
329    205116
313    204098
312    198899
324    181870
317    180961
331    177924
310    172430
325    144175
318    142129
304    137804
311    134496
332      6067
Name: count, dtype: int64

In [37]:
df.sample(1000).to_csv("sample.csv", index=False)

In [35]:
df["time"]

0         2016-11-13 05:00:00
1         2016-11-13 05:00:00
2         2016-11-13 05:00:00
3         2016-11-13 05:00:00
4         2016-11-13 05:00:00
                  ...        
5465570   2016-11-05 20:00:00
5465571   2016-11-05 20:00:00
5465572   2016-11-05 20:00:00
5465573   2016-11-05 19:50:00
5465574   2016-11-05 19:50:00
Name: time, Length: 5465575, dtype: datetime64[s]

In [32]:
df.nunique()

delay            4082
vehicle_type        3
line_number        67
direction           2
stop_id          1530
weekday             7
time             3526
temp              143
windspeed_max     131
windspeed_avg      66
precipitation      15
dew_point         120
humidity           46
hour               23
dayminute         132
dtype: int64

In [ ]:
# Use if needed to get see all cols of pandas dataframes
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
df.head()

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

# -- For IID Data
splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)

# -- For Grouped Non-IID data
splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)

# -- For Temporal non-IID data -> manual processing required
# splits = {
#     repeat_i: {
#         fold_i: (train_idx, test_idx),
#     }
# }

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

## Export

In [ ]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)